# Consultor Financeiro Inteligente - Quantum Finance

In [33]:
# Instalação das dependências necessárias
#!pip install langchain langchain-openai langgraph mcp duckduckgo-search nest-asyncio pandas requests pywin32

## Configuração de Credenciais e Introdução ao MCP

In [ ]:
import getpass
import os
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import json
from typing import Annotated, List, Dict, Any
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver


# Habilitar suporte a loops assíncronos no Jupyter Notebook
nest_asyncio.apply()

# Configurando as chaves de API no ambiente
os.environ["OPENAI_API_KEY"] = ""

os.environ["BOLSAI_API_KEY"] = ""


## Implementação do Cliente MCP em Python

In [35]:
class BolsaiMCPClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        # Configurar parâmetros do servidor MCP usando o uvx para execução direta sem precisar instalar nada globalmente
        self.server_params = StdioServerParameters(
            command="uvx",
            args=["bolsai-mcp"],
            env={"BOLSAI_API_KEY": self.api_key}
        )
        self.read_stream = None
        self.write_stream = None
        self.session = None
        self._client_context = None

    async def connect(self):
        """Conecta ao servidor MCP e inicializa a sessão"""
        import sys
        errlog = sys.stderr
        if hasattr(sys, "__stderr__") and sys.__stderr__ is not None:
            try:
                sys.__stderr__.fileno()
                errlog = sys.__stderr__
            except Exception:
                pass
        self._client_context = stdio_client(self.server_params, errlog=errlog)
        self.read_stream, self.write_stream = await self._client_context.__aenter__()
        self.session = ClientSession(self.read_stream, self.write_stream)
        await self.session.__aenter__()
        await self.session.initialize()
        print("Conectado com sucesso ao Bolsai MCP Server!")

    async def disconnect(self):
        """Finaliza a conexão e limpa os recursos"""
        try:
            if self.session:
                await self.session.__aexit__(None, None, None)
        except Exception:
            pass
        try:
            if self._client_context:
                await self._client_context.__aexit__(None, None, None)
        except Exception:
            pass
        print("Desconectado do Bolsai MCP Server.")

    async def call_tool(self, tool_name: str, arguments: dict):
        """Executa uma ferramenta específica no servidor MCP e retorna o JSON/texto"""
        if not self.session:
            raise RuntimeError("Cliente MCP não conectado. Chame connect() primeiro.")
        result = await self.session.call_tool(tool_name, arguments)
        if not result.content:
            return "Nenhum dado retornado."
        return result.content[0].text


## Inicialização do Cliente MCP Global

In [36]:
# Instanciar e conectar ao cliente global
bolsai_client = BolsaiMCPClient(api_key=os.environ["BOLSAI_API_KEY"])

loop = asyncio.get_event_loop()
loop.run_until_complete(bolsai_client.connect())

Task exception was never retrieved
future: <Task finished name='Task-391' coro=<<async_generator_athrow without __name__>()> exception=RuntimeError('Attempted to exit cancel scope in a different task than it was entered in')>
  + Exception Group Traceback (most recent call last):
  |   File "c:\Users\stgab\OneDrive\Documentos\GitHub\Grupo_2_atividades\.venv\Lib\site-packages\anyio\_backends\_asyncio.py", line 799, in __aexit__
  |     raise BaseExceptionGroup(
  | BaseExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)
  +-+---------------- 1 ----------------
    | Traceback (most recent call last):
    |   File "c:\Users\stgab\OneDrive\Documentos\GitHub\Grupo_2_atividades\.venv\Lib\site-packages\mcp\client\stdio\__init__.py", line 189, in stdio_client
    |     yield read_stream, write_stream
    | GeneratorExit
    +------------------------------------

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Use

Conectado com sucesso ao Bolsai MCP Server!


## Definição das Ferramentas (Tools) da B3 e Busca Web

In [37]:
# Ferramenta de busca externa para conceitos financeiros
web_search = DuckDuckGoSearchRun()

@tool
def search_financial_concept(query: str) -> str:
    """Busca na internet informações conceituais sobre produtos e regulamentos financeiros (ex: como funciona o Tesouro Direto, CDB, LCI, LCA).
    Use esta ferramenta quando o cliente tiver dúvidas de finanças básicas ou estruturais.
    """
    try:
        return web_search.run(query)
    except Exception as e:
        return f"Erro na busca: {str(e)}"

# Ferramentas da B3 conectadas ao Bolsai MCP Server
@tool
def get_stock_quote(ticker: str) -> str:
    """Busca a cotação em tempo real de uma ação ou FII na B3 (preço atual, variação, volume)."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(bolsai_client.call_tool("get_stock_quote", {"ticker": ticker.upper()}))

@tool
def get_fundamentals(ticker: str) -> str:
    """Busca múltiplos fundamentalistas (P/L, P/VP, ROE, Margem Líquida, Dívida Líquida/EBITDA) de uma ação da B3."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(bolsai_client.call_tool("get_fundamentals", {"ticker": ticker.upper()}))

@tool
def get_dividends(ticker: str) -> str:
    """Busca informações de dividendos (Dividend Yield, histórico recente de provisões) de um ativo da B3."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(bolsai_client.call_tool("get_dividends", {"ticker": ticker.upper()}))

@tool
def get_macro_data(indicator: str) -> str:
    """Busca o valor atual e histórico recente de indicadores macroeconômicos brasileiros (selic, cdi, ipca, usdbrl)."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(bolsai_client.call_tool("get_macro_data", {"indicator": indicator.lower()}))

@tool
def compare_stocks(tickers: list) -> str:
    """Compara múltiplos fundamentalistas de até 5 tickers de ações da B3 lado a lado."""
    loop = asyncio.get_event_loop()
    tickers_upper = [t.upper() for t in tickers]
    return loop.run_until_complete(bolsai_client.call_tool("compare_stocks", {"tickers": tickers_upper}))

@tool
def get_fii_data(ticker: str) -> str:
    """Busca indicadores específicos de Fundos Imobiliários (FIIs) como P/VP, Dividend Yield 12m e segmento de atuação."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(bolsai_client.call_tool("get_fii_data", {"ticker": ticker.upper()}))

@tool
def screen_stocks(roe_min: float = None, pl_max: float = None, sector: str = None) -> str:
    """Filtra e retorna uma lista de ações da B3 que atendem a múltiplos financeiros específicos (ex: roe_min=15, pl_max=12)."""
    loop = asyncio.get_event_loop()
    args = {}
    if roe_min is not None: args["roe_min"] = roe_min
    if pl_max is not None: args["pl_max"] = pl_max
    if sector is not None: args["sector"] = sector
    return loop.run_until_complete(bolsai_client.call_tool("screen_stocks", args))

## Configuração da Estrutura de Agentes (LangGraph)

In [38]:
# Estado compartilhado entre os agentes
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    user_profile: Dict[str, Any]

# Definição das ferramentas de delegação para o Estrategista (Lead Advisor)
@tool
def consult_market_analyst(query: str) -> str:
    """Delega uma tarefa ao Agente Pesquisador (Market Analyst).
    Use esta ferramenta para buscar explicações conceituais de produtos financeiros (CDB, FIIs, LCI, Tesouro Direto, etc.) na internet.
    """
    pass

@tool
def consult_b3_specialist(task: str) -> str:
    """Delega uma tarefa ao Especialista B3.
    Use esta ferramenta para buscar dados reais da bolsa (cotações, múltiplos de ações, dividendos, indicadores macroeconômicos).
    """
    pass

# Inicialização dos modelos OpenAI
advisor_model = ChatOpenAI(model="gpt-4o", temperature=0.2).bind_tools([
    consult_market_analyst,
    consult_b3_specialist
])

# Prompt de Sistema do Lead Advisor
ADVISOR_SYSTEM_PROMPT = (
    "Você é o Lead Advisor (Estrategista Financeiro) da Quantum Finance.\n"
    "Seu papel é receber as dúvidas do cliente, entender o perfil dele, planejar a busca de dados reais "
    "e delegar tarefas para seus subagentes usando as ferramentas:\n"
    "- 'consult_market_analyst' para dúvidas teóricas ou conceitos.\n"
    "- 'consult_b3_specialist' para dados de mercado, cotações e múltiplos da B3.\n\n"
    "Perfil do Cliente Atual:\n"
    "{user_profile}\n\n"
    "Diretrizes:\n"
    "1. Nunca tente chutar ou alucinar cotações e múltiplos. Sempre chame o consult_b3_specialist.\n"
    "2. Quando tiver todos os dados necessários reunidos, elabore um relatório final com a recomendação de alocação personalizada, "
    "justificando cada indicação de acordo com o perfil do cliente e os múltiplos da B3 de 2026 obtidos."
)


## Implementação dos Nós do Grafo

In [39]:
# 1. Nó do Lead Advisor
def lead_advisor_node(state: AgentState):
    profile_str = "\n".join([f"- {k}: {v}" for k, v in state["user_profile"].items()])
    sys_msg = SystemMessage(content=ADVISOR_SYSTEM_PROMPT.format(user_profile=profile_str))
    
    # Executar o modelo do estrategista
    response = advisor_model.invoke([sys_msg] + state["messages"])
    return {"messages": [response]}

# 2. Nó Executor de Subagentes (Market Analyst e Especialista B3)
def subagents_executor_node(state: AgentState):
    last_message = state["messages"][-1]
    new_messages = []
    
    for tool_call in last_message.tool_calls:
        if tool_call["name"] == "consult_market_analyst":
            query = tool_call["args"]["query"]
            print(f"\n[Nó Ativo: subagents_executor -> Executando Market Analyst para a busca: '{query}']")
            
            analyst_model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools([search_financial_concept])
            analyst_system = (
                "Você é o Agente Pesquisador (Market Analyst). Sua tarefa é explicar de forma didática conceitos financeiros.\n"
                "Use a busca na internet para responder."
            )
            analyst_messages = [SystemMessage(content=analyst_system), HumanMessage(content=query)]
            response = analyst_model.invoke(analyst_messages)
            
            while response.tool_calls:
                analyst_messages.append(response)
                for tc in response.tool_calls:
                    result = search_financial_concept.invoke(tc["args"])
                    analyst_messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
                response = analyst_model.invoke(analyst_messages)
                
            tool_msg = ToolMessage(content=response.content, tool_call_id=tool_call["id"], name=tool_call["name"])
            new_messages.append(tool_msg)
            print(f"-> Resposta do Market Analyst obtida.")
            
        elif tool_call["name"] == "consult_b3_specialist":
            task = tool_call["args"]["task"]
            print(f"\n[Nó Ativo: subagents_executor -> Executando Agente B3 para a tarefa: '{task}']")
            
            b3_tools = [
                get_stock_quote,
                get_fundamentals,
                get_dividends,
                get_macro_data,
                compare_stocks,
                get_fii_data,
                screen_stocks
            ]
            b3_model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(b3_tools)
            b3_system = (
                "Você é o Agente de Dados B3. Sua tarefa é extrair múltiplos e cotações da B3."
            )
            b3_messages = [SystemMessage(content=b3_system), HumanMessage(content=task)]
            response = b3_model.invoke(b3_messages)
            
            while response.tool_calls:
                b3_messages.append(response)
                for tc in response.tool_calls:
                    target_tool = next(t for t in b3_tools if t.name == tc["name"])
                    result = target_tool.invoke(tc["args"])
                    b3_messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
                response = b3_model.invoke(b3_messages)
                
            tool_msg = ToolMessage(content=response.content, tool_call_id=tool_call["id"], name=tool_call["name"])
            new_messages.append(tool_msg)
            print(f"-> Resposta do Agente B3 obtida.")
            
    return {"messages": new_messages}


## Definição de Rotas e Compilação do Grafo

In [40]:
def route_after_advisor(state: AgentState):
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return END
    return "subagents_executor"

# Construção do Grafo
memory = MemorySaver()
workflow = StateGraph(AgentState)

workflow.add_node("lead_advisor", lead_advisor_node)
workflow.add_node("subagents_executor", subagents_executor_node)

workflow.add_edge(START, "lead_advisor")

# Rota condicional a partir do Lead Advisor
workflow.add_conditional_edges(
    "lead_advisor",
    route_after_advisor,
    {
        "subagents_executor": "subagents_executor",
        END: END
    }
)

# Retorna o controle para o Lead Advisor planejar o próximo passo
workflow.add_edge("subagents_executor", "lead_advisor")

app = workflow.compile(checkpointer=memory)
print("Grafo compilado com sucesso com checkpointer de memória!")


Grafo compilado com sucesso com checkpointer de memória!


## Execução de Testes Práticos (Simulações de Casos de Uso)

In [41]:
def start_chat(perfil: dict):
    """Inicia um chat conversacional interativo com histórico de mensagens usando o checkpointer do LangGraph"""
    print("=== INICIANDO CONSULTORIA FINANCEIRA INTERATIVA ===")
    print(f"Perfil do Cliente: {perfil}")
    print("Digite 'sair' para encerrar a conversa.\n")
    
    config = {"configurable": {"thread_id": "consultoria_client_thread"}, "recursion_limit": 15}
    
    while True:
        try:
            pergunta = input("Você: ")
        except KeyboardInterrupt:
            print("\nSaindo...")
            break
            
        if pergunta.lower().strip() in ["sair", "exit", "quit"]:
            print("Consultoria finalizada. Até mais!")
            break
            
        if not pergunta.strip():
            continue
            
        print("\n[Consultando agentes e processando resposta...]")
        
        for event in app.stream({"messages": [HumanMessage(content=pergunta)], "user_profile": perfil}, config):
            for node_name, state_update in event.items():
                messages = state_update.get("messages", [])
                for msg in messages:
                    if isinstance(msg, AIMessage):
                        if msg.tool_calls:
                            for tc in msg.tool_calls:
                                print(f" -> [Estrategista] Delegando: {tc['name']} com parâmetros {tc['args']}")
                        else:
                            print(f"\n[Relatório Final]:\n{msg.content}\n")
                    elif isinstance(msg, ToolMessage):
                        print(f" -> [Subagente] Retornou resposta.")


In [42]:
# Caso de Teste: Gabriel, investidor Arrojado buscando alocação de longo prazo e cotações
perfil_arrojado = {
    "Nome": "Gabriel",
    "Perfil de Risco": "Arrojado",
    "Objetivo": "Maximizar ganhos em renda variável no longo prazo",
    "Horizonte de Tempo": "10 anos"
}

# Chat conversacional
start_chat(perfil=perfil_arrojado)


=== INICIANDO CONSULTORIA FINANCEIRA INTERATIVA ===
Perfil do Cliente: {'Nome': 'Gabriel', 'Perfil de Risco': 'Arrojado', 'Objetivo': 'Maximizar ganhos em renda variável no longo prazo', 'Horizonte de Tempo': '10 anos'}
Digite 'sair' para encerrar a conversa.


[Consultando agentes e processando resposta...]

[Relatório Final]:
Olá, Gabriel! Eu sou o Lead Advisor da Quantum Finance, e estou aqui para ajudar você a maximizar seus ganhos em renda variável no longo prazo. Posso responder suas dúvidas sobre conceitos financeiros, buscar dados de mercado, cotações e múltiplos da B3, e elaborar um relatório de alocação personalizada de acordo com seu perfil de risco arrojado e seu horizonte de tempo de 10 anos.

Se você tiver alguma dúvida ou precisar de informações específicas, é só me avisar!


[Consultando agentes e processando resposta...]
 -> [Estrategista] Delegando: consult_market_analyst com parâmetros {'query': 'O que são FIIs (Fundos de Investimento Imobiliário)?'}

[Nó Ativo: suba

## Finalização da Conexão MCP

In [43]:
# Desconectar do servidor MCP
loop.run_until_complete(bolsai_client.disconnect())

Desconectado do Bolsai MCP Server.
